# Compare the models

Every model in `MODELS` is run on the same images and scored by the **same code**
(`train/metrics.py`, `train/analysis.py`), whatever produced it. Boxes are compared in
original-image pixels.

Protocol (see README):
- each model keeps its own post-processing: the NMS IoU threshold is **tuned per model**
  on this split, and each model's confidence operating point is its own best-F1 one;
- mAP uses conf 0.001 (it sweeps over confidence; the cutoff only drops negligible boxes);
- `SPLIT = "valid"` for analysis. **The test split is for the final numbers only** --
  every choice made from this notebook's output (NMS, thresholds, which model) must come
  from val first.

Sections 6-11 only need the cached predictions from section 6, so they can be rerun
freely. Runs on CPU too (slower).

## 1. Settings + mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/yolo-edge-quantization"
MANIFEST_RUN = "run04_long"   # the split all models were trained on
SPLIT = "valid"               # "test" only for the final comparison

# add a model = add a line. kind: "own" (this repo) or "ultralytics"
MODELS = [
    {"name": "own (run04)", "kind": "own", "weights": f"{RUN_ROOT}/run04_long/best.pt"},
    {"name": "yolov8n scratch", "kind": "ultralytics", "weights": f"{RUN_ROOT}/yolov8n_scratch/train/weights/best.pt"},
    # {"name": "yolov8n pretrained", "kind": "ultralytics", "weights": f"{RUN_ROOT}/yolov8n_pretrained/train/weights/best.pt"},
]

IMG_SIZE = 320
NMS_CANDIDATES = (0.5, 0.6, 0.7)   # tuned per model on SPLIT
STRICT_IOU = False                  # also mAP@[.5:.95] -- ~10x slower (plain-Python mAP)

## 2. Download the dataset

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")

## 3. Clone (or update) the repo, install Ultralytics

In [ ]:
import os
import sys

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

!pip install -q -r {repoDirectory}/requirements.txt
!pip install -q ultralytics==8.3.40
!git -C {repoDirectory} log --oneline -1

sys.path.insert(0, repoDirectory)

## 4. Load the split

`gtSizes`: each object's size as the models see it -- longest side in 320x320
model-input pixels (the frame the own model's size thresholds 32/80 are defined in).

In [ ]:
import json

import torch

from train.analysis import image_sizes, manifest_gts

with open(f"{RUN_ROOT}/{MANIFEST_RUN}/manifest.json") as f:
    manifest = json.load(f)
with open(f"{RUN_ROOT}/{MANIFEST_RUN}/classes.json") as f:
    classNames = json.load(f)
numClasses = len(classNames)

if SPLIT == "test":
    print("*** TEST SPLIT -- final numbers only; make no decisions from these results ***")

samples = manifest[SPLIT]
gts = manifest_gts(samples)
sizes = image_sizes(samples)
gtSizes = [
    torch.maximum((b[:, 2] - b[:, 0]) * IMG_SIZE / w, (b[:, 3] - b[:, 1]) * IMG_SIZE / h)
    for (b, _), (w, h) in zip(gts, sizes)
]

trainCounts = [0] * numClasses
for s in manifest["train"]:
    for label in s["labels"]:
        trainCounts[label] += 1
splitCounts = [0] * numClasses
for _, labels in gts:
    for label in labels.tolist():
        splitCounts[label] += 1

print(f"{SPLIT}: {len(samples)} images, {sum(splitCounts)} objects")

## 5. Load the models: size and compute

GFLOPs at 320x320, counted by PyTorch's FLOP counter for both models (2 FLOPs per
multiply-add).

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
from ultralytics import YOLO as UltralyticsYOLO

from models.yolo import YOLO
from train.trainer import load_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"

for entry in MODELS:
    if entry["kind"] == "own":
        net = YOLO(numClasses).to(device)
        load_checkpoint(entry["weights"], net, device=device)
        entry["model"], module = net, net
    else:
        entry["model"] = UltralyticsYOLO(entry["weights"])
        module = entry["model"].model.to(device)
    module.eval()

    entry["params"] = sum(p.numel() for p in module.parameters())
    with FlopCounterMode(display=False) as counter, torch.no_grad():
        module(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device))
    entry["gflops"] = counter.get_total_flops() / 1e9
    print(f"{entry['name']:22s} {entry['params']:>10,} params   {entry['gflops']:.2f} GFLOPs")

## 6. Predictions, with the NMS threshold tuned per model

Each model is run once per candidate NMS IoU; the one with the best mAP@0.5 on this
split is kept (`entry["preds"]`) for every section below.

In [ ]:
import pandas as pd

from train.analysis import predict_own_model
from train.metrics import mean_average_precision


def predict(entry, nmsIou):
    if entry["kind"] == "own":
        return predict_own_model(entry["model"], samples, IMG_SIZE, device, conf_thresh=0.001, nms_iou=nmsIou)
    preds = []
    for start in range(0, len(samples), 32):
        batch = samples[start:start + 32]
        results = entry["model"].predict([s["image_path"] for s in batch], imgsz=IMG_SIZE, conf=0.001,
                                         iou=nmsIou, max_det=300, verbose=False)
        preds += [(r.boxes.xyxy.cpu(), r.boxes.conf.cpu(), r.boxes.cls.cpu().long()) for r in results]
    return preds


sweep = []
for entry in MODELS:
    best = None
    for nmsIou in NMS_CANDIDATES:
        preds = predict(entry, nmsIou)
        mAP = mean_average_precision(preds, gts, numClasses)[0.5]
        sweep.append({"model": entry["name"], "NMS IoU": nmsIou, "mAP@0.5": mAP})
        if best is None or mAP > best[0]:
            best = (mAP, nmsIou, preds)
    entry["map50"], entry["nmsIou"], entry["preds"] = best

pd.DataFrame(sweep).pivot(index="model", columns="NMS IoU", values="mAP@0.5").round(4)

## 7. Headline numbers

- **class-agnostic mAP@0.5**: every label set to one class -- "found the object, box in
  the right place". Its gap to mAP@0.5 is what classification mistakes cost each model.
- **best-F1 operating point**: the confidence threshold maximizing F1 over all boxes --
  where each model would naturally be deployed; used for sections 9-11.

In [ ]:
from train.analysis import best_f1


def agnostic(preds, gts):
    return ([(b, s, torch.zeros_like(l)) for b, s, l in preds], [(b, torch.zeros_like(l)) for b, l in gts])


rows = []
for entry in MODELS:
    agnosticMap = mean_average_precision(*agnostic(entry["preds"], gts), 1)[0.5]
    entry["op"], _ = best_f1(entry["preds"], gts)
    row = {
        "model": entry["name"], "params (M)": entry["params"] / 1e6, "GFLOPs@320": entry["gflops"],
        "NMS IoU": entry["nmsIou"], "mAP@0.5": entry["map50"],
        "class-agnostic mAP@0.5": agnosticMap, "cost of classification": agnosticMap - entry["map50"],
        "best-F1 conf": entry["op"]["conf"], "precision": entry["op"]["precision"],
        "recall": entry["op"]["recall"], "F1": entry["op"]["f1"],
    }
    if STRICT_IOU:
        thresholds = tuple(round(0.5 + 0.05 * i, 2) for i in range(10))
        row["mAP@[.5:.95]"] = mean_average_precision(entry["preds"], gts, numClasses, thresholds)["mAP@[.5:.95]"]
    rows.append(row)

pd.DataFrame(rows).set_index("model").round(4).T

## 8. Per-class AP

AP@0.5 for every class, next to how many training examples it had. If one model's lead
comes from rare classes, it shows here (e.g. the effect of repeat-factor sampling, which
only the own model uses). Classes with no object in this split have no AP.

In [ ]:
import matplotlib.pyplot as plt

from train.analysis import per_class_ap

perClass = pd.DataFrame({"class": classNames, "train objects": trainCounts, f"{SPLIT} objects": splitCounts})
for entry in MODELS:
    perClass[entry["name"]] = per_class_ap(entry["preds"], gts, numClasses)
names = [entry["name"] for entry in MODELS]
if len(names) >= 2:
    perClass[f"{names[0]} - {names[1]}"] = perClass[names[0]] - perClass[names[1]]

pd.set_option("display.max_rows", 100)
perClass.sort_values("train objects").round(3).reset_index(drop=True)

In [ ]:
# AP vs. training examples
figure, axis = plt.subplots(figsize=(9, 5))
for name in names:
    axis.scatter(perClass["train objects"], perClass[name], label=name, alpha=0.7)
axis.set_xscale("log")
axis.set_xlabel("training objects of the class (log scale)")
axis.set_ylabel(f"AP@0.5 on {SPLIT}")
axis.set_ylim(-0.05, 1.05)
axis.legend()
plt.show()

# mean AP per class-frequency band
bands = [0, 20, 100, 500, float("inf")]
perClass["train objects band"] = pd.cut(perClass["train objects"], bands, right=False,
                                        labels=["<20", "20-99", "100-499", ">=500"])
banded = perClass.groupby("train objects band", observed=False)[names].mean()
banded.insert(0, "classes", perClass.groupby("train objects band", observed=False)["class"].count())
banded.round(3)

## 9. Recall by object size

Share of objects found, by size (longest side in 320x320 model-input pixels), at conf
0.001 (the most a model can find at all) and at its best-F1 operating point.

In [ ]:
from train.analysis import recall_by_size

sizeEdges = (16, 32, 64, 128)
rows = []
for entry in MODELS:
    for label, conf in (("max (conf 0.001)", 0.001), ("at best-F1 conf", entry["op"]["conf"])):
        for band, n, recall in recall_by_size(entry["preds"], gts, gtSizes, sizeEdges, conf):
            rows.append({"model": entry["name"], "recall": label, "size (px)": band, "objects": n, "value": recall})

bySize = pd.DataFrame(rows)
print("objects per size band:", dict(bySize.drop_duplicates("size (px)")[["size (px)", "objects"]].values))
bySize.pivot_table(index=["model", "recall"], columns="size (px)", values="value", sort=False).round(3)

## 10. What each model confuses

Boxes that found a real object (IoU >= 0.5, any class) but gave it the wrong class, at
each model's best-F1 operating point.

In [ ]:
from train.analysis import confusion_pairs

for entry in MODELS:
    pairs = confusion_pairs(entry["preds"], gts, entry["op"]["conf"])
    top = sorted(pairs.items(), key=lambda kv: -kv[1])[:15]
    print(f"--- {entry['name']}: {sum(pairs.values())} wrong-class detections, top {len(top)}")
    for (gtClass, predClass), count in top:
        print(f"  {count:3d} x  {classNames[gtClass]:28s} -> predicted {classNames[predClass]}")
    print()

## 11. Side by side

The same random images, one column per model, each at its own best-F1 confidence.
Green = ground truth, red = predictions. Rerun for a new draw; set `seed` to reproduce.

In [ ]:
import random

from PIL import Image

from train.visualize import draw_detections

nImages = 4
seed = None
indices = random.Random(seed).sample(range(len(samples)), min(nImages, len(samples)))
print(f"{SPLIT} indices:", indices)

figure, axes = plt.subplots(len(indices), len(MODELS), figsize=(6 * len(MODELS), 5 * len(indices)), squeeze=False)
for row, idx in enumerate(indices):
    image = Image.open(samples[idx]["image_path"]).convert("RGB")
    gtBoxes, gtLabels = gts[idx]
    for col, entry in enumerate(MODELS):
        boxes, scores, labels = entry["preds"][idx]
        keep = scores >= entry["op"]["conf"]
        drawn = draw_detections(image, gtBoxes, gtLabels, class_names=classNames, color="green")
        drawn = draw_detections(drawn, boxes[keep], labels[keep], scores=scores[keep],
                                class_names=classNames, color="red")
        axes[row, col].imshow(drawn)
        axes[row, col].axis("off")
        axes[row, col].set_title(f"{entry['name']} (conf {entry['op']['conf']}): "
                                 f"{int(keep.sum())} predicted, {len(gtBoxes)} GT")
plt.tight_layout()
plt.show()